In [ ]:
import requests
import tiktoken
import torch
from torch import nn
import datasets


tokenizer = tiktoken.get_encoding("gpt2")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [ ]:
engine_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/model.py")

with open("model.py", "w") as f:
    f.write(engine_res.text)
    # print(engine_res.text)
print("Downloaded model.py")

# Downloading train_utils.py
train_utils_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/train_utils.py")

with open("train_utils.py", "w") as f:
    f.write(train_utils_res.text)
    # print(engine_res.text)
print("Downloaded train_utils.py")

data_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/data.py")

with open("data.py", "w") as f:
    f.write(data_res.text)
    # print(engine_res.text)
print("Downloaded data.py")

#Downloading custom_gpt_v2.pth weights
wt_res = requests.get("https://github.com/Shanmukh-dev/Custom-GPT/raw/refs/heads/main/Models/tara_n1_pretrain_v2.pth")
with open("tara_n1_pretrain_v2.pth", "wb") as f:
    f.write(wt_res.content)
print("Downloaded modle weights")


Downloaded model.py
Downloaded train_utils.py
Downloaded data.py
Downloaded modle weights


In [ ]:
from model import *
from train_utils import *
from data import *


config = GPTConfig(
    vocab_size=tokenizer.n_vocab,
    block_size=1024,
    batch_size=4,
    d_model=256,
    hidden_layers=1024,
    n_heads=4,
    n_layers=6,
)

# Resasoninng instruciton tuning using **gsm8k** dataset

## Download and prep Dataset


In [ ]:
from datasets import load_dataset

ds = load_dataset("SimpleStories/SimpleStories")

print(len(ds))

README.md:   0%|          | 0.00/4.18k [00:00<?, ?B/s]

data/train-00000-of-00007.parquet: reconstructing file:   0%|          |  0.00B /  238MB            

data/train-00000-of-00007.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00007.parquet: reconstructing file:   0%|          |  0.00B /  238MB            

data/train-00001-of-00007.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 16.8MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2115696 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/21371 [00:00<?, ? examples/s]

2


In [ ]:
len(ds["train"])

2115696

In [ ]:
ds["train"][0]

{'story': "Eagerly, a girl named Kim went hiking with her friends. They found a secret passageway under a big rock. Curious, they crawled through it and entered a magical world where all the animals talked. They learned that the animals had their own struggles but always found a way to be happy. \n\nInspired, the children played with the animals and shared their own stories of hardship. They realized that laughter could help them through hard times. When they finally left, they took the animals' smiles with them, knowing that happiness can come from the most unexpected places.",
 'topic': 'subterranean worlds',
 'theme': 'Hardship',
 'style': 'lighthearted',
 'feature': 'symbolism',
 'grammar': '',
 'persona': '',
 'initial_word_type': 'adverb',
 'initial_letter': 'T',
 'word_count': 109,
 'character_count': 570,
 'num_paragraphs': 2,
 'avg_word_length': 4.34,
 'avg_sentence_length': 15.57,
 'flesch_reading_ease': 74.49,
 'flesch_kincaid_grade': 6.3,
 'dale_chall_readability_score': 8.

### Creating dataloaders

In [ ]:
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
class GSMDataset(Dataset):
    def __init__(self, data, block_size):
        self.data = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        x = self.data[idx:idx+self.block_size]
        y = self.data[idx+1:idx+self.block_size+1]

        return x, y


def create_data_doc(data):
    doc = ""
    pbar = tqdm(total=len(data))
    for sample in data:
        text = sample["story"]
        doc += f"{text}\n{tokenizer.decode([tokenizer.eot_token])}\n\n"
        pbar.update(1)


    return doc.strip()

In [ ]:
ds["train"][0]["story"]+tokenizer.decode([tokenizer.eot_token])

"Eagerly, a girl named Kim went hiking with her friends. They found a secret passageway under a big rock. Curious, they crawled through it and entered a magical world where all the animals talked. They learned that the animals had their own struggles but always found a way to be happy. \n\nInspired, the children played with the animals and shared their own stories of hardship. They realized that laughter could help them through hard times. When they finally left, they took the animals' smiles with them, knowing that happiness can come from the most unexpected places.<|endoftext|>"

In [ ]:
train_data = torch.tensor(tokenizer.encode(create_data_doc(ds["train"]), allowed_special="all"), dtype=torch.long, device=device)
test_data = torch.tensor(tokenizer.encode(create_data_doc(ds["test"]), allowed_special="all"), dtype=torch.long, device=device)

train_dataset = GSMDataset(train_data, config.block_size)
test_dataset = GSMDataset(test_data, config.block_size)

train_dataloader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=True)

In [ ]:
print(len(test_dataset))

84458


## Training loop

In [ ]:
model = CustomGPT(config)
state_dict = torch.load("tara_n1_pretrain_v2.pth", map_location=device)
state_dict = {k.removeprefix("module."):v for k, v in state_dict.items()}
model.load_state_dict(state_dict)
model.to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler()

In [ ]:
from tqdm.auto import tqdm

steps = 2065
test_count = 500
test_steps = 20

train_iter = iter(train_dataloader)

avg_train_loss = 0

for step in tqdm(range(1, steps+1)):
    model.train()
    if not train_iter:
        train_iter = iter(train_dataloader)

    train_loss = train_step(model, train_iter, loss_fn, optimizer, scaler, device, accumulation_steps=1)
    avg_train_loss += train_loss
    if step%test_count == 0:
        avg_train_loss /= test_count

        test_loss = test_step(model, test_dataloader, test_steps, loss_fn, device)

        print(f"Step: {step} | Train loss: {avg_train_loss} | Test loss: {test_loss}")
        avg_train_loss = 0


  0%|          | 0/2065 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
model.eval()

query = "Once upon a time, there live a king "
context = torch.tensor(tokenizer.encode(query), dtype=torch.long, device=device).unsqueeze(0)

with torch.inference_mode():
    generated = model.generate(context)

print(tokenizer.decode(generated[0].tolist()).split("<|endoftext|>")[0].replace("&amp;", "&").replace("&lt;", "<").replace("&gt;", ">"))


Once upon a time, there live a king ________________________Middle School smell.(hi josa˚m Ãuinge de los Reyes>
https 2023 Monday morning dust bulbs
https 2023 Monday morning dust bulbs
https 3027 Friday gloom bulbs
https 1029 Friday spellings
https 1223Thursday morning dust bulbs
https 1284 Saturdaydays face indime
https tip ornamentacles
https lengthens typches
https skyrocket
https plant trunks
https images Jobs
https nutscomb
https summer lapers
birdacle yellow

 Lakes errs
https
invariation in redwood
https download Brooklyn
https flyered
https landscapes
https follow shrubs
https summer l Publications
httpsqueries
quality | Spring trunks
https English
https little screen prints
https past
https archives
https catalogue deserts
https ninety20 resolutions
https paste wps
https blindness marks
https text irides
https garden dream
https wallpapers
https pages Highlands
https catalog deserts
https mailcolor coverage
https partisans
https archlibrary pages
https new pages Visiting
http